# Block 3 — persona checks

Verifies the synthetic respondents produced by `pipeline/make_personas.py`
(`pipeline/personas.csv`).

Checks:
1. **Shape & balance** — N per condition × 17 conditions; correct columns.
2. **Validity** — unique `profile_id`; demographic values are valid schema codes;
   `year_birth` implies an adult age (18–100).
3. **Marginals match the cited targets** — empirical shares ≈ the ACS/Gallup/Pew
   distributions wired into the generator (within sampling tolerance).
4. **Provenance recorded** — `demographics_sources.md` documents the sources.
5. **Integration test** — personas + random answers → real `clean.R` → demographics
   decode to valid labels, `age_band` is built, all 17 conditions survive.

In [ ]:
import csv, json, re, shutil, subprocess, sys, tempfile, random
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "codebook.csv").exists():
    ROOT = ROOT.parent
assert (ROOT / "codebook.csv").exists(), f"can't find repo root from cwd={Path.cwd()}"
PIPE = ROOT / "pipeline"
sys.path.insert(0, str(PIPE))
print("repo root:", ROOT)

failures = []
def check(name, ok, detail=""):
    print(f"[{'PASS' if ok else 'FAIL'}] {name}" + (f"  -- {detail}" if (detail and not ok) else ""))
    if not ok:
        failures.append((name, detail))

## Step 0 — (Re)build and load personas

In [ ]:
import make_personas
N = 50
rows = make_personas.build_personas(N)                    # regenerate in-memory
with (PIPE / "personas.csv").open(newline="", encoding="utf-8") as f:
    disk = list(csv.DictReader(f))
schema = json.loads((PIPE / "schema.json").read_text(encoding="utf-8"))
print(f"{len(rows)} personas in-memory; {len(disk)} on disk; N per condition = {N}")

## Check 1 — Shape & balance

In [ ]:
from collections import Counter
check("personas.csv columns == generator schema", list(disk[0]) == make_personas.PERSONA_COLUMNS,
      f"{list(disk[0])}")
conds = schema["conditions"]["all"]
per_cond = Counter(r["condition"] for r in disk)
check("row count == N x 17", len(disk) == N * len(conds), f"got {len(disk)}")
check("balanced: exactly N per condition", set(per_cond.values()) == {N} and set(per_cond) == set(conds),
      f"counts={dict(per_cond)}")

## Check 2 — Validity

In [ ]:
ids = [r["profile_id"] for r in disk]
check("profile_id unique", len(set(ids)) == len(ids))

code_sets = {d["qualtrics_label"]: set(d["scale"]["codes"])
             for d in schema["demographics"] if d["scale"]["kind"] == "categorical"}
bad_codes = {}
for name, valid in code_sets.items():
    seen = {r[name] for r in disk}
    if not seen <= valid:
        bad_codes[name] = seen - valid
check("demographic values are valid schema codes", not bad_codes, str(bad_codes))

ages = [2026 - int(r["year_birth"]) for r in disk]
check("year_birth implies adult age 18-100", all(18 <= a <= 100 for a in ages),
      f"min={min(ages)}, max={max(ages)}")

## Check 3 — Empirical marginals match the cited targets

In [ ]:
def emp(name):
    c = Counter(r[name] for r in disk)
    return {k: c.get(k, 0) / len(disk) for k in code_sets[name]}

def ageband(a):
    return "18-29" if a <= 29 else "30-44" if a <= 44 else "45-59" if a <= 59 else "60+"

TOL = 0.06
worst = 0.0
for name, target in make_personas.MARGINALS.items():
    e = emp(name)
    tot = sum(target.values())
    for code_, p in target.items():
        dev = abs(e[code_] - p / tot)
        worst = max(worst, dev)
        check(f"{name}[{code_}] within {TOL} of target", dev < TOL, f"emp={e[code_]:.3f} target={p/tot:.3f}")

# age band (derived from year_birth)
ab = Counter(ageband(a) for a in ages)
tot = sum(make_personas.AGE_BAND_MARGINALS.values())
for band, p in make_personas.AGE_BAND_MARGINALS.items():
    dev = abs(ab[band]/len(ages) - p/tot)
    worst = max(worst, dev)
    check(f"age_band[{band}] within {TOL} of target", dev < TOL,
          f"emp={ab[band]/len(ages):.3f} target={p/tot:.3f}")
print(f"worst deviation from target: {worst:.3f}  (sampling noise at N={len(disk)})")

## Check 4 — Provenance recorded

In [ ]:
src = (PIPE / "demographics_sources.md")
check("demographics_sources.md exists", src.exists())
if src.exists():
    t = src.read_text(encoding="utf-8")
    check("sources doc cites ACS, Gallup, and Pew",
          all(s in t for s in ["ACS 2023", "Gallup", "Pew Research"]))

## Check 5 — Integration test: personas + answers → real clean.R

In [ ]:
rscript = shutil.which("Rscript")
if not rscript:
    print("[skip] Rscript not found -- run locally to exercise clean.R.")
else:
    # ground truth from R
    r_code = """suppressMessages(source('scripts/lib/submission_spec.R')); library(jsonlite)
      cat(toJSON(list(conditions=sst$conditions, moderators=sst$moderators,
                      tier1_required=sst$tier1_required), auto_unbox=TRUE))"""
    R = json.loads(subprocess.run([rscript, "-e", r_code], cwd=ROOT,
                                  capture_output=True, text=True, check=True).stdout)

    raw_cols = schema["raw_export_columns"]
    elicited = schema["elicited_items"]
    random.seed(1)
    def answer(it):
        k = it["scale"]["kind"]
        if k == "integer": return str(random.randint(0, 10))
        if k == "binary":  return random.choice(["Yes", "No"])
        return str(random.randint(0, 100))
    persona_by_col = {c for c in make_personas.PERSONA_COLUMNS}

    full_rows = []
    for p in disk:
        row = dict(p)                                  # the 8 assigned fields
        for it in elicited:
            row[it["qualtrics_label"]] = answer(it)    # fill the 44 model items
        full_rows.append(row)

    with tempfile.TemporaryDirectory() as td:
        ip, op = Path(td) / "raw.csv", Path(td) / "clean.csv"
        with ip.open("w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=raw_cols); w.writeheader(); w.writerows(full_rows)
        res = subprocess.run([rscript, "scripts/clean.R", str(ip), str(op)],
                             cwd=ROOT, capture_output=True, text=True)
        check("clean.R runs on personas + answers (exit 0)", res.returncode == 0,
              (res.stderr or "").strip()[-800:])
        if res.returncode == 0:
            with op.open(encoding="utf-8") as f:
                out = list(csv.DictReader(f))
            check("row count preserved through cleaning", len(out) == len(disk), f"{len(out)} vs {len(disk)}")
            check("all 17 conditions survive", {r["condition"] for r in out} == set(R["conditions"]))
            mods = R["moderators"]
            demo_ok = {}
            for name in ["gender", "race", "education", "income", "party"]:
                vals = {r[name] for r in out}
                if not vals <= set(mods[name]):
                    demo_ok[name] = vals - set(mods[name])
            check("demographics decoded to valid labels (no bad/NA)", not demo_ok, str(demo_ok))
            ab_vals = {r["age_band"] for r in out}
            check("age_band built and valid", ab_vals <= set(mods["age_band"]), str(ab_vals))

## Summary

In [ ]:
print("=" * 60)
if failures:
    print(f"BLOCK 3 PERSONAS: {len(failures)} CHECK(S) FAILED")
    for n, d in failures:
        print("  -", n, "::", d)
else:
    print("BLOCK 3 PERSONAS: ALL CHECKS PASSED  \u2705")
assert not failures, f"{len(failures)} check(s) failed"